<a href="https://colab.research.google.com/github/Rundrayer/Evaluacion_Practica_Grupal_IDS221/blob/main/Calculo_Deficit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Punto 1
import pandas as pd

operaciones = pd.read_csv("operacion_sedes.csv")
sedes = pd.read_csv("sedes_geograficas.csv")


operaciones["deficit_kwh"] = (
    operaciones["demanda_kwh"] - operaciones["generacion_kwh"]
).clip(lower=0)

#  Sumar el deficit acumulado de cada sede
deficit_sedes = operaciones.groupby(
    "sede_id", as_index=False
)["deficit_kwh"].sum()

#  Cambiar el nombre de la columna del deficit acumulado
deficit_sedes = deficit_sedes.rename(
    columns={"deficit_kwh": "deficit_total_kwh"}
)

# 5. Asociar el deficit con los datos geograficos
resultado = pd.merge(
    sedes,
    deficit_sedes,
    on="sede_id",
    how="left"
)

print("Deficit acumulado por sede:")
print(resultado.to_string(index=False))

#  Mostrar las sedes ordenadas de mayor a menor deficit
print("\nSedes ordenadas por deficit:")
print(
    resultado.sort_values(
        "deficit_total_kwh", ascending=False
    )[["sede_id", "nombre_sede", "deficit_total_kwh"]]
    .to_string(index=False)
)


Deficit acumulado por sede:
sede_id        nombre_sede    municipio    region  latitud  longitud  deficit_total_kwh
    S01   Solar Ahuachapán   Ahuachapán Occidente  13.9214  -89.8453           36692.27
    S02    Solar Santa Ana    Santa Ana Occidente  13.9942  -89.5597           20465.48
    S03    Solar Sonsonate    Sonsonate Occidente  13.7189  -89.7241             747.15
    S04 Solar San Salvador San Salvador    Centro  13.6929  -89.2182           77336.41
    S05  Solar Santa Tecla  Santa Tecla    Centro  13.6769  -89.2797           25528.84
    S06  Solar Cojutepeque  Cojutepeque    Centro  13.7179  -88.9331           10909.28
    S07   Solar San Miguel   San Miguel   Oriente  13.4837  -88.1779           69364.49
    S08     Solar Usulután     Usulután   Oriente  13.3500  -88.4461            3593.47
    S09     Solar La Unión     La Unión   Oriente  13.3369  -67.8440           42266.81

Sedes ordenadas por deficit:
sede_id        nombre_sede  deficit_total_kwh
    S04 Solar Sa

In [ ]:
#Punto 2
import pandas as pd

sedes = pd.read_csv("sedes_geograficas.csv")

# Convertir las coordenadas a valores numéricos
sedes["latitud"] = pd.to_numeric(
    sedes["latitud"], errors="coerce"
)

sedes["longitud"] = pd.to_numeric(
    sedes["longitud"], errors="coerce"
)

# Verificar la plausibilidad territorial
sedes["coordenadas_validas"] = (
    sedes["latitud"].between(13, 15)
    & sedes["longitud"].between(-91, -87)
)

# Separar las sedes válidas e inválidas
sedes_mapa = sedes[
    sedes["coordenadas_validas"]
].copy()

sedes_invalidas = sedes[
    ~sedes["coordenadas_validas"]
].copy()

print("SEDES VÁLIDAS PARA EL MAPA:")
print(sedes_mapa.to_string(index=False))

print("\nSEDES EXCLUIDAS DEL MAPA:")
print(sedes_invalidas.to_string(index=False))

print("\nRESUMEN:")
print("Total de sedes:", len(sedes))
print("Sedes válidas:", len(sedes_mapa))
print("Sedes excluidas:", len(sedes_invalidas))


SEDES VÁLIDAS PARA EL MAPA:
sede_id        nombre_sede    municipio    region  latitud  longitud  coordenadas_validas
    S01   Solar Ahuachapán   Ahuachapán Occidente  13.9214  -89.8453                 True
    S02    Solar Santa Ana    Santa Ana Occidente  13.9942  -89.5597                 True
    S03    Solar Sonsonate    Sonsonate Occidente  13.7189  -89.7241                 True
    S04 Solar San Salvador San Salvador    Centro  13.6929  -89.2182                 True
    S05  Solar Santa Tecla  Santa Tecla    Centro  13.6769  -89.2797                 True
    S06  Solar Cojutepeque  Cojutepeque    Centro  13.7179  -88.9331                 True
    S07   Solar San Miguel   San Miguel   Oriente  13.4837  -88.1779                 True
    S08     Solar Usulután     Usulután   Oriente  13.3500  -88.4461                 True

SEDES EXCLUIDAS DEL MAPA:
sede_id    nombre_sede municipio  region  latitud  longitud  coordenadas_validas
    S09 Solar La Unión  La Unión Oriente  13.3369   -6

In [ ]:

import pandas as pd
from google.colab import files

operaciones = pd.read_csv("operacion_sedes.csv")
sedes = pd.read_csv("sedes_geograficas.csv")

# 2. Calcular el déficit por registro
operaciones["deficit_kwh"] = (
    operaciones["demanda_kwh"] -
    operaciones["generacion_kwh"]
).clip(lower=0)

# 3. Calcular el déficit acumulado por sede
deficit_sedes = (
    operaciones.groupby("sede_id")["deficit_kwh"]
    .sum()
    .round(2)
    .reset_index(name="deficit_total_kwh")
)

# 4. Convertir las coordenadas a valores numéricos
sedes["latitud"] = pd.to_numeric(
    sedes["latitud"], errors="coerce"
)
sedes["longitud"] = pd.to_numeric(
    sedes["longitud"], errors="coerce"
)

# 5. Asociar los datos mediante sede_id
geo = sedes.merge(
    deficit_sedes,
    on="sede_id",
    how="left"
)

# 6. Validar las coordenadas
validas = (
    geo["latitud"].between(13, 15)
    & geo["longitud"].between(-91, -87)
    & geo["deficit_total_kwh"].notna()
)

# 7. Seleccionar las columnas solicitadas
geo_validado = geo.loc[
    validas,
    ["sede_id", "latitud", "longitud", "deficit_total_kwh"]
].copy()

# 8. Exportar el CSV con dos decimales
geo_validado.to_csv(
    "geo_validado.csv",
    index=False,
    float_format="%.2f"
)


# 10. Descargar el archivo
files.download("geo_validado.csv")



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>